In [1]:
import glob
import json
import os
from google.colab import drive
import pandas as pd
import requests

# 0. Asegurar que Google Drive esté montado
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

os.makedirs('data/raw', exist_ok=True)

# 1. Cargar datos académicos reales y filtrar los 4 campus principales
BASE_PATH = '/content/drive/MyDrive/PROYECTO ADQUISICIÓN DE DATOS/PROYECTO FINAL/Fuentes de datos/Datos académicos ITM/Datos anonimizados'

# Diagnóstico de ruta
if not os.path.exists(BASE_PATH):
    raise FileNotFoundError(
        f'La ruta no existe. Revisa la escritura exacta:\n{BASE_PATH}'
    )

archivos = sorted(glob.glob(os.path.join(BASE_PATH, 'Procesado_*.xlsx')))

if not archivos:
    contenido = os.listdir(BASE_PATH)
    raise FileNotFoundError(
        "No se encontraron archivos 'Procesado_*.xlsx'. "
        f'Archivos encontrados en la carpeta: {contenido}'
    )

print(f'Se encontraron {len(archivos)} archivos:')
for a in archivos:
    print(' -', os.path.basename(a))


Mounted at /content/drive
Se encontraron 4 archivos:
 - Procesado_2022-1.xlsx
 - Procesado_2022-2.xlsx
 - Procesado_2023-1.xlsx
 - Procesado_2023-2.xlsx


In [2]:
import os
import glob
import json
import requests
import pandas as pd

# Crear carpeta de datos crudos
os.makedirs('data/raw', exist_ok=True)

# 1. Cargar datos académicos reales y filtrar los 4 campus principales
BASE_PATH = '/content/drive/MyDrive/PROYECTO ADQUISICIÓN DE DATOS/PROYECTO FINAL/Fuentes de datos/Datos académicos ITM/Datos anonimizados'
archivos = sorted(glob.glob(os.path.join(BASE_PATH, 'Procesado_*.xlsx')))

dfs = []
for f in archivos:
    df_t = pd.read_excel(f)
    df_t['periodo_archivo'] = os.path.basename(f).replace('Procesado_', '').replace('.xlsx', '')
    dfs.append(df_t)

df_acad = pd.concat(dfs, ignore_index=True)

# Normalizar nombre de campus para hacer match exacto con las 4 sedes
def mapear_campus(val):
    s = str(val).upper()
    if 'FRATERNIDAD' in s: return 'FRATERNIDAD'
    if 'ROBLEDO' in s: return 'ROBLEDO'
    if 'CASTILLA' in s: return 'CASTILLA'
    if 'C4TA' in s: return 'CAMPUS C4TA'
    return 'OTRO'

df_acad['campus_norm'] = df_acad['Campus'].apply(mapear_campus)
df_acad_sedes = df_acad[df_acad['campus_norm'] != 'OTRO'].copy()

print("Distribución de estudiantes en las 4 sedes principales:")
print(df_acad_sedes['campus_norm'].value_counts())

# 2. Coordenadas exactas de las 4 sedes
COORDENADAS_ITM = {
    'FRATERNIDAD': {'lat': 6.245607756232677,  'lon': -75.55088302377642},
    'ROBLEDO':     {'lat': 6.2737230974828995, 'lon': -75.58886669121688},
    'CASTILLA':    {'lat': 6.293956803108489,  'lon': -75.5692246964889},
    'CAMPUS C4TA': {'lat': 6.253579693991596,  'lon': -75.61330516672112}
}

# 3. Consultar API Open-Meteo por cada sede y consolidar en un único JSON crudo
URL_API = "https://archive-api.open-meteo.com/v1/archive"
api_sedes_json = {
    "metadata": {
        "fuente": "Open-Meteo Historical Weather API",
        "endpoint": URL_API,
        "rango": "2022-01-01 a 2023-12-31",
        "timezone": "America/Bogota"
    },
    "sedes": {}
}

for campus, coords in COORDENADAS_ITM.items():
    params = {
        "latitude": coords['lat'],
        "longitude": coords['lon'],
        "start_date": "2022-01-01",
        "end_date": "2023-12-31",
        "hourly": ["temperature_2m", "precipitation", "rain"],
        "timezone": "America/Bogota"
    }
    resp = requests.get(URL_API, params=params, timeout=30)
    resp.raise_for_status()
    api_sedes_json["sedes"][campus] = resp.json()
    print(f"Descargado clima para {campus}: {len(resp.json()['hourly']['time'])} registros horarios.")

ruta_json = 'data/raw/open_meteo_sedes_raw.json'
with open(ruta_json, 'w', encoding='utf-8') as f:
    json.dump(api_sedes_json, f, ensure_ascii=False, indent=2)

print(f"\nFuente API guardada exitosamente en: {ruta_json}")

Distribución de estudiantes en las 4 sedes principales:
campus_norm
FRATERNIDAD    1437
ROBLEDO         455
CASTILLA         82
CAMPUS C4TA      53
Name: count, dtype: int64
Descargado clima para FRATERNIDAD: 17520 registros horarios.
Descargado clima para ROBLEDO: 17520 registros horarios.
Descargado clima para CASTILLA: 17520 registros horarios.
Descargado clima para CAMPUS C4TA: 17520 registros horarios.

Fuente API guardada exitosamente en: data/raw/open_meteo_sedes_raw.json


In [3]:
import json
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

# 1. Cargar el JSON crudo de Open-Meteo (precipitación + temperatura por sede y hora)
with open('data/raw/open_meteo_sedes_raw.json', 'r', encoding='utf-8') as f:
    clima_json = json.load(f)

lluvia_lookup = {}
temp_lookup = {}
for campus, payload in clima_json['sedes'].items():
    df_c = pd.DataFrame(payload['hourly'])
    df_c['time'] = pd.to_datetime(df_c['time'])
    for row_c in df_c.itertuples():
        clave = (campus, str(row_c.time.date()), row_c.time.hour)
        lluvia_lookup[clave] = float(row_c.precipitation or 0.0)
        temp_lookup[clave] = float(row_c.temperature_2m or 19.0)

# 2. Calendarios de 17 semanas (119 días) proyectados a 2022 y 2023 según estructura ITM
SEMESTRES_17_SEMANAS = {
    '2022-1': ('2022-02-07', '2022-06-05'),
    '2022-2': ('2022-08-01', '2022-11-27'),
    '2023-1': ('2023-02-06', '2023-06-04'),
    '2023-2': ('2023-07-31', '2023-11-26')
}

SEMANAS_PARCIALES = {4, 8, 13, 17}

BLOQUES_HORARIOS = [
    ([1, 3], 6),   # Martes y Jueves 06:00 - 08:00
    ([0, 2], 8),   # Lunes y Miércoles 08:00 - 10:00
    ([1, 3], 14),  # Martes y Jueves 14:00 - 16:00
    ([2, 4], 18),  # Miércoles y Viernes 18:00 - 20:00
]

PREFIJO_AULA = {
    'FRATERNIDAD': 'F',
    'ROBLEDO': 'R',
    'CASTILLA': 'C',
    'CAMPUS C4TA': 'T'
}

registros_sensor = []

for idx, row in df_acad_sedes.iterrows():
    est_id = str(row['ID_Anonimizado']).strip()
    campus = row['campus_norm']
    periodo = row['periodo_archivo']
    resultado = str(row['Resultado']).strip()
    nota_final = pd.to_numeric(row['Final'], errors='coerce')
    nota_final = 2.5 if pd.isna(nota_final) else float(nota_final)

    if periodo not in SEMESTRES_17_SEMANAS:
        continue

    inicio, fin = SEMESTRES_17_SEMANAS[periodo]
    dias_sem, hora_inicio = BLOQUES_HORARIOS[idx % len(BLOQUES_HORARIOS)]
    aula = f"{PREFIJO_AULA[campus]}-{101 + (idx % 12)}"

    fechas_sem = pd.date_range(inicio, fin, freq='D')
    fechas_clase = [d for d in fechas_sem if d.weekday() in dias_sem]

    # Probabilidad base vinculada al desempeño real en 'Final'
    prob_base = np.clip(0.28 - (nota_final * 0.045), 0.03, 0.40)
    semana_cancelacion = int(rng.integers(12, 16)) if 'Cancel' in resultado else 99

    for clase_idx, fecha_dt in enumerate(fechas_clase):
        num_semana = (clase_idx // 2) + 1
        es_parcial = num_semana in SEMANAS_PARCIALES
        fecha_str = str(fecha_dt.date())

        # Ventana climática de 3 horas (desplazamiento previo + inicio) y temperatura
        lluvia_m2 = lluvia_lookup.get((campus, fecha_str, max(0, hora_inicio - 2)), 0.0)
        lluvia_m1 = lluvia_lookup.get((campus, fecha_str, max(0, hora_inicio - 1)), 0.0)
        lluvia_0  = lluvia_lookup.get((campus, fecha_str, hora_inicio), 0.0)
        lluvia_3h = lluvia_m2 + lluvia_m1 + lluvia_0
        temp_act  = temp_lookup.get((campus, fecha_str, hora_inicio), 19.0)

        if num_semana >= semana_cancelacion:
            prob_falta = 0.94  # Abandonó tras cancelar materia
        else:
            factor_estacional = -0.06 if es_parcial else (0.04 if num_semana >= 14 and nota_final < 3.0 else 0.0)

            # Curva continua logarítmica + penalización por tormenta fuerte (> 5.0 mm en ERA5)
            efecto_lluvia = (
                0.05 * np.log1p(lluvia_3h * 2.5)
                + (0.22 if lluvia_3h > 5.0 else (0.08 if lluvia_3h > 1.5 else 0.0))
            )
            # En semana de parciales los estudiantes resisten más el mal clima
            if es_parcial:
                efecto_lluvia *= 0.55

            # Penalización térmica cuando hace frío (< 17 °C, típico de madrugadas lluviosas)
            efecto_frio = max(0.0, (17.0 - temp_act) * 0.022)

            prob_falta = np.clip(
                prob_base + factor_estacional + efecto_lluvia + efecto_frio,
                0.02,
                0.90
            )

        falto = rng.random() < prob_falta

        if falto:
            registros_sensor.append({
                'id_evento': f"EVT-{periodo}-{idx}-{clase_idx}",
                'id_estudiante': est_id,
                'campus': campus,
                'periodo': periodo,
                'semana_semestre': num_semana,
                'es_semana_parcial': es_parcial,
                'aula': aula,
                'fecha': fecha_str,
                'bloque_hora': f"{hora_inicio:02d}:00",
                'timestamp_entrada': pd.NaT,
                'timestamp_salida': pd.NaT,
                'estado_asistencia': 'AUSENTE',
                'rssi_dbm': np.nan
            })
        else:
            retraso_min = int(np.clip(rng.normal(loc=2 + (lluvia_3h * 1.6), scale=5), -10, 45))
            ts_in = pd.Timestamp(fecha_str) + pd.Timedelta(hours=hora_inicio, minutes=retraso_min)
            ts_out = ts_in + pd.Timedelta(minutes=int(rng.integers(95, 115)))
            estado = 'TARDE' if retraso_min > 15 else 'PRESENTE'

            registros_sensor.append({
                'id_evento': f"EVT-{periodo}-{idx}-{clase_idx}",
                'id_estudiante': est_id,
                'campus': campus,
                'periodo': periodo,
                'semana_semestre': num_semana,
                'es_semana_parcial': es_parcial,
                'aula': aula,
                'fecha': fecha_str,
                'bloque_hora': f"{hora_inicio:02d}:00",
                'timestamp_entrada': ts_in,
                'timestamp_salida': ts_out,
                'estado_asistencia': estado,
                'rssi_dbm': round(float(rng.uniform(-65.0, -42.0)), 1)
            })

df_sensor = pd.DataFrame(registros_sensor)

# 3. Inyección controlada de errores (duplicados, nulos y anomalías físicas)
n_total = len(df_sensor)
idx_dup = rng.choice(df_sensor.index, size=int(n_total * 0.008), replace=False)
df_sensor = pd.concat([df_sensor, df_sensor.loc[idx_dup]], ignore_index=True)

idx_null = rng.choice(df_sensor.index, size=int(n_total * 0.006), replace=False)
df_sensor.loc[idx_null, 'estado_asistencia'] = np.nan

idx_anom = rng.choice(df_sensor[df_sensor['estado_asistencia'] == 'PRESENTE'].index, size=int(n_total * 0.003), replace=False)
df_sensor.loc[idx_anom, 'rssi_dbm'] = 18.5

# 4. Exportar a formato Parquet
ruta_parquet = 'data/raw/sensor_rfid_aula_raw.parquet'
df_sensor.to_parquet(ruta_parquet, engine='pyarrow', index=False)

# 5. AUDITORÍA DE CORRELACIÓN CLIMA vs. ASISTENCIA (Verificación rápida)
df_val = df_sensor.dropna(subset=['estado_asistencia']).drop_duplicates(subset=['id_evento']).copy()
df_val['asistio'] = df_val['estado_asistencia'].isin(['PRESENTE', 'TARDE']).astype(int)
df_val['hora_int'] = df_val['bloque_hora'].str.slice(0, 2).astype(int)
df_val['lluvia_3h'] = [
    sum(lluvia_lookup.get((r.campus, r.fecha, max(0, r.hora_int - k)), 0.0) for k in (0, 1, 2))
    for r in df_val.itertuples()
]

resumen_bloques = (
    df_val.groupby(['campus', 'fecha', 'bloque_hora'], as_index=False)
    .agg(tasa_asistencia=('asistio', 'mean'), lluvia_3h=('lluvia_3h', 'first'), n=('asistio', 'count'))
)
resumen_bloques = resumen_bloques[resumen_bloques['n'] >= 15]

r_pearson = resumen_bloques['lluvia_3h'].corr(resumen_bloques['tasa_asistencia'], method='pearson')
r_spearman = resumen_bloques['lluvia_3h'].corr(resumen_bloques['tasa_asistencia'], method='spearman')

print(f"Fuente Sensor guardada en: {ruta_parquet} ({df_sensor.shape[0]} filas)")
print(f"Correlación lluvia_3h vs tasa_asistencia -> Pearson r = {r_pearson:.3f} | Spearman rho = {r_spearman:.3f}")
print("\nTop 5 bloques con peor lluvia y su caída de asistencia:")
print(resumen_bloques.sort_values('lluvia_3h', ascending=False).head(5).to_string(index=False))

Fuente Sensor guardada en: data/raw/sensor_rfid_aula_raw.parquet (69469 filas)
Correlación lluvia_3h vs tasa_asistencia -> Pearson r = -0.519 | Spearman rho = -0.468

Top 5 bloques con peor lluvia y su caída de asistencia:
     campus      fecha bloque_hora  tasa_asistencia  lluvia_3h   n
    ROBLEDO 2023-10-03       14:00         0.459459       15.2  37
    ROBLEDO 2022-05-19       14:00         0.100000       14.3  20
    ROBLEDO 2023-10-12       14:00         0.432432       14.2  37
    ROBLEDO 2023-05-16       14:00         0.285714       14.0  35
FRATERNIDAD 2023-10-03       14:00         0.394958       13.0 119
